# Лабораторная работа 2: Классификация текстов с применением RNN, LSTM, GRU

**Задание:**
- Классификация текстов (просьбы жильцов) с помощью RNN, LSTM, GRU
- Сравнение качества трёх моделей
- Реализация собственного GRU-блока с нуля

**Источники:** `materials/Лаб2.md`

## 1. Импорты и настройка окружения

Загружаем необходимые библиотеки для работы с PyTorch, данными и метриками.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import matplotlib.pyplot as plt
import re
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Загрузка и предобработка датасета

Загружаем датасет "просьбы жильцов" (Petitions.csv), выполняем предварительную обработку текста: приведение к нижнему регистру, удаление спецсимволов, токенизация. Датасет большой, возьмём часть для работы (п.1-2).

In [ ]:
# Загрузка датасета
df = pd.read_csv('../lab1/datasets/Petitions.csv')
print(f"Исходный размер датасета: {len(df)}")
print(f"Столбцы: {df.columns.tolist()}")

# Возьмём часть датасета для работы (например, 5000 строк)
SAMPLE_SIZE = 5000
df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)
print(f"Размер выборки: {len(df)}")

# Определяем столбцы с текстом и целевым признаком
text_col = df.columns[0]  # Первый столбец - текст
target_col = df.columns[1] if len(df.columns) > 1 else None  # Второй столбец - категория (если есть)

print(f"\nСтолбец с текстом: {text_col}")
print(f"Столбец с категорией: {target_col}")

# Если целевой признак отсутствует, создадим синтетический (по длине текста)
if target_col is None or df[target_col].isna().all():
    print("Целевой признак отсутствует, создаём синтетический по длине текста")
    df['category'] = pd.cut(df[text_col].str.len(), bins=3, labels=['короткий', 'средний', 'длинный'])
    target_col = 'category'

print(f"\nРаспределение категорий:\n{df[target_col].value_counts()}")

### Предобработка текста

Функция для очистки текста: приведение к нижнему регистру, удаление спецсимволов, токенизация по пробелам.

In [ ]:
def preprocess_text(text):
    """Предобработка текста: нижний регистр, удаление спецсимволов, токенизация"""
    if pd.isna(text):
        return []
    # Нижний регистр
    text = str(text).lower()
    # Удаление спецсимволов (оставляем буквы, цифры, пробелы)
    text = re.sub(r'[^a-zA-Zа-яА-ЯёЁ0-9\s]', '', text)
    # Токенизация по пробелам
    tokens = text.split()
    return tokens

# Применяем предобработку
df['tokens'] = df[text_col].apply(preprocess_text)

# Статистика по длине токенизированных текстов
df['token_length'] = df['tokens'].apply(len)
print(f"Средняя длина токенов: {df['token_length'].mean():.2f}")
print(f"Максимальная длина токенов: {df['token_length'].max()}")
print(f"Минимальная длина токенов: {df['token_length'].min()}")

# Удаляем пустые тексты
df = df[df['token_length'] > 0].reset_index(drop=True)
print(f"\nРазмер после удаления пустых текстов: {len(df)}")

## 3. Токенизация и векторизация

Создаём словарь (vocabulary) на основе токенов корпуса. Ограничиваем размер словаря наиболее частыми словами. Векторизуем каждый токен: используем простую индексацию (embedding обучается в модели). Можно использовать обученные Word2Vec модели из ЛР 1 (п.3).

In [ ]:
# Параметры векторизации
VOCAB_SIZE = 10000  # Размер словаря (наиболее частые слова)
MAX_SEQ_LENGTH = 50  # Максимальная длина последовательности (паддинг/обрезка)

# Сбор всех токенов
all_tokens = [token for tokens in df['tokens'] for token in tokens]
print(f"Всего токенов в корпусе: {len(all_tokens)}")

# Создание словаря (частотный)
word_counts = Counter(all_tokens)
print(f"Уникальных токенов: {len(word_counts)}")

# Словарь: слово -> индекс (оставляем VOCAB_SIZE-1 наиболее частых + PAD и UNK)
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, count in word_counts.most_common(VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

print(f"Размер словаря: {len(vocab)}")

# Функция для преобразования токенов в индексы
def tokens_to_indices(tokens, vocab, max_length):
    """Преобразует список токенов в список индексов с паддингом/обрезкой"""
    indices = [vocab.get(token, vocab['<UNK>']) for token in tokens]
    # Обрезка или паддинг до max_length
    if len(indices) > max_length:
        indices = indices[:max_length]
    else:
        indices = indices + [vocab['<PAD>']] * (max_length - len(indices))
    return indices

# Применяем векторизацию
df['indices'] = df['tokens'].apply(lambda x: tokens_to_indices(x, vocab, MAX_SEQ_LENGTH))

print(f"Пример векторизованной последовательности: {df['indices'][0][:10]}...")
print(f"Длина последовательности: {len(df['indices'][0])}")

## 4. Кодирование целевого признака и разделение на train/test

Кодируем категории с помощью LabelEncoder. Разделяем данные на обучающую и тестовую выборки (п.4).

In [ ]:
# Кодирование целевого признака
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df[target_col])

print(f"Классы: {label_encoder.classes_}")
print(f"Количество классов: {len(label_encoder.classes_)}")
print(f"Распределение меток:\n{df['label'].value_counts()}")

# Разделение на train/test
X = np.array(df['indices'].tolist())
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nРазмер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

## 5. PyTorch Dataset и DataLoader

Создаём класс TextDataset для работы с данными в PyTorch. Настраиваем DataLoader для batch-обучения.

In [ ]:
class TextDataset(Dataset):
    """Dataset для текстовых данных"""
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Создание Dataset и DataLoader
BATCH_SIZE = 32

train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Размер train dataset: {len(train_dataset)}")
print(f"Размер test dataset: {len(test_dataset)}")
print(f"Количество batch в train: {len(train_loader)}")
print(f"Количество batch в test: {len(test_loader)}")

## 6. Модель на базе RNN

Создаём нейросеть с рекуррентным блоком RNN для классификации текстов (п.5). Архитектура: Embedding → RNN → Fully Connected → Softmax.

In [ ]:
class RNNClassifier(nn.Module):
    """Классификатор на базе RNN"""
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers=1):
        super(RNNClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # x: [batch_size, seq_length]
        embedded = self.embedding(x)  # [batch_size, seq_length, embedding_dim]
        output, hidden = self.rnn(embedded)  # output: [batch_size, seq_length, hidden_dim], hidden: [num_layers, batch_size, hidden_dim]
        # Берём скрытое состояние последнего временного шага
        hidden = hidden[-1]  # [batch_size, hidden_dim]
        logits = self.fc(hidden)  # [batch_size, output_dim]
        return logits

# Параметры модели
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
OUTPUT_DIM = len(label_encoder.classes_)
NUM_LAYERS = 2

rnn_model = RNNClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, NUM_LAYERS)
print(rnn_model)

### Обучение RNN модели

Функция обучения с отслеживанием loss по эпохам.

In [ ]:
def train_model(model, train_loader, criterion, optimizer, num_epochs):
    """Обучение модели"""
    model.train()
    train_losses = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")
    
    return train_losses

# Обучение RNN
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(rnn_model.parameters(), lr=0.001)
NUM_EPOCHS = 20

print("Обучение RNN модели...")
rnn_losses = train_model(rnn_model, train_loader, criterion, optimizer, NUM_EPOCHS)

### Оценка качества RNN модели

Вычисляем метрики на тестовой выборке: accuracy, precision, recall, F1.

In [ ]:
def evaluate_model(model, test_loader):
    """Оценка модели на тестовой выборке"""
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = model(batch_X)
            _, preds = torch.max(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    
    return accuracy, precision, recall, f1, all_labels, all_preds

# Оценка RNN
rnn_acc, rnn_prec, rnn_rec, rnn_f1, rnn_labels, rnn_preds = evaluate_model(rnn_model, test_loader)

print(f"RNN - Accuracy: {rnn_acc:.4f}")
print(f"RNN - Precision: {rnn_prec:.4f}")
print(f"RNN - Recall: {rnn_rec:.4f}")
print(f"RNN - F1: {rnn_f1:.4f}")
print("\nClassification Report:")
print(classification_report(rnn_labels, rnn_preds, target_names=label_encoder.classes_))

## 7. Модель на базе LSTM

Создаём нейросеть с рекуррентным блоком LSTM для классификации текстов (п.5). Архитектура аналогична RNN, но с LSTM вместо RNN.

In [ ]:
class LSTMClassifier(nn.Module):
    """Классификатор на базе LSTM"""
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers=1):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # x: [batch_size, seq_length]
        embedded = self.embedding(x)  # [batch_size, seq_length, embedding_dim]
        output, (hidden, cell) = self.lstm(embedded)  # hidden: [num_layers, batch_size, hidden_dim]
        # Берём скрытое состояние последнего временного шага
        hidden = hidden[-1]  # [batch_size, hidden_dim]
        logits = self.fc(hidden)  # [batch_size, output_dim]
        return logits

# Создание и обучение LSTM модели
lstm_model = LSTMClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, NUM_LAYERS)
print(lstm_model)

optimizer_lstm = optim.Adam(lstm_model.parameters(), lr=0.001)

print("\nОбучение LSTM модели...")
lstm_losses = train_model(lstm_model, train_loader, criterion, optimizer_lstm, NUM_EPOCHS)

### Оценка качества LSTM модели

In [ ]:
# Оценка LSTM
lstm_acc, lstm_prec, lstm_rec, lstm_f1, lstm_labels, lstm_preds = evaluate_model(lstm_model, test_loader)

print(f"LSTM - Accuracy: {lstm_acc:.4f}")
print(f"LSTM - Precision: {lstm_prec:.4f}")
print(f"LSTM - Recall: {lstm_rec:.4f}")
print(f"LSTM - F1: {lstm_f1:.4f}")
print("\nClassification Report:")
print(classification_report(lstm_labels, lstm_preds, target_names=label_encoder.classes_))

## 8. Модель на базе GRU

Создаём нейросеть с рекуррентным блоком GRU для классификации текстов (п.5). Архитектура аналогична RNN/LSTM, но с GRU.

In [ ]:
class GRUClassifier(nn.Module):
    """Классификатор на базе GRU"""
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers=1):
        super(GRUClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # x: [batch_size, seq_length]
        embedded = self.embedding(x)  # [batch_size, seq_length, embedding_dim]
        output, hidden = self.gru(embedded)  # hidden: [num_layers, batch_size, hidden_dim]
        # Берём скрытое состояние последнего временного шага
        hidden = hidden[-1]  # [batch_size, hidden_dim]
        logits = self.fc(hidden)  # [batch_size, output_dim]
        return logits

# Создание и обучение GRU модели
gru_model = GRUClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, NUM_LAYERS)
print(gru_model)

optimizer_gru = optim.Adam(gru_model.parameters(), lr=0.001)

print("\nОбучение GRU модели...")
gru_losses = train_model(gru_model, train_loader, criterion, optimizer_gru, NUM_EPOCHS)

### Оценка качества GRU модели

In [ ]:
# Оценка GRU
gru_acc, gru_prec, gru_rec, gru_f1, gru_labels, gru_preds = evaluate_model(gru_model, test_loader)

print(f"GRU - Accuracy: {gru_acc:.4f}")
print(f"GRU - Precision: {gru_prec:.4f}")
print(f"GRU - Recall: {gru_rec:.4f}")
print(f"GRU - F1: {gru_f1:.4f}")
print("\nClassification Report:")
print(classification_report(gru_labels, gru_preds, target_names=label_encoder.classes_))

## 9. Сравнение качества моделей

Сравниваем метрики трёх моделей (RNN, LSTM, GRU) и делаем вывод о том, какая модель лучше (п.6).

In [ ]:
# Сводная таблица метрик
comparison = pd.DataFrame({
    'Model': ['RNN', 'LSTM', 'GRU'],
    'Accuracy': [rnn_acc, lstm_acc, gru_acc],
    'Precision': [rnn_prec, lstm_prec, gru_prec],
    'Recall': [rnn_rec, lstm_rec, gru_rec],
    'F1': [rnn_f1, lstm_f1, gru_f1]
})

print("Сравнение качества моделей:")
print(comparison.to_string(index=False))

# График сравнения
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
for i, metric in enumerate(metrics):
    axes[i].bar(comparison['Model'], comparison[metric], color=['blue', 'green', 'red'])
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    axes[i].set_ylabel('Score')
plt.tight_layout()
plt.show()

# Вывод о лучшей модели
best_model = comparison.loc[comparison['F1'].idxmax(), 'Model']
print(f"\nЛучшая модель по F1: {best_model}")

## 10. Реализация собственного GRU-блока

Реализуем GRU-блок с нуля используя PyTorch (п.7). GRU (Gated Recurrent Unit) содержит update gate и reset gate для управления потоком информации.

In [ ]:
class CustomGRUCell(nn.Module):
    """Собственная реализация GRU-ячейки"""
    def __init__(self, input_dim, hidden_dim):
        super(CustomGRUCell, self).__init__()
        self.hidden_dim = hidden_dim
        
        # Reset gate: r_t = sigmoid(W_ir * x_t + b_ir + W_hr * h_{t-1} + b_hr)
        self.W_ir = nn.Linear(input_dim, hidden_dim)
        self.W_hr = nn.Linear(hidden_dim, hidden_dim)
        
        # Update gate: z_t = sigmoid(W_iz * x_t + b_iz + W_hz * h_{t-1} + b_hz)
        self.W_iz = nn.Linear(input_dim, hidden_dim)
        self.W_hz = nn.Linear(hidden_dim, hidden_dim)
        
        # Candidate hidden state: h_tilde = tanh(W_ih * x_t + b_ih + r_t * (W_hh * h_{t-1} + b_hh))
        self.W_ih = nn.Linear(input_dim, hidden_dim)
        self.W_hh = nn.Linear(hidden_dim, hidden_dim)
        
    def forward(self, x, h_prev):
        """
        x: [batch_size, input_dim]
        h_prev: [batch_size, hidden_dim]
        """
        # Reset gate
        r = torch.sigmoid(self.W_ir(x) + self.W_hr(h_prev))
        
        # Update gate
        z = torch.sigmoid(self.W_iz(x) + self.W_hz(h_prev))
        
        # Candidate hidden state
        h_tilde = torch.tanh(self.W_ih(x) + r * self.W_hh(h_prev))
        
        # New hidden state
        h = (1 - z) * h_prev + z * h_tilde
        
        return h

class CustomGRULayer(nn.Module):
    """Слой собственной GRU для обработки последовательностей"""
    def __init__(self, input_dim, hidden_dim):
        super(CustomGRULayer, self).__init__()
        self.hidden_dim = hidden_dim
        self.gru_cell = CustomGRUCell(input_dim, hidden_dim)
        
    def forward(self, x):
        """
        x: [batch_size, seq_length, input_dim]
        """
        batch_size, seq_length, _ = x.size()
        
        # Инициализация скрытого состояния
        h = torch.zeros(batch_size, self.hidden_dim, device=x.device)
        
        # Обработка последовательности по временным шагам
        outputs = []
        for t in range(seq_length):
            h = self.gru_cell(x[:, t, :], h)
            outputs.append(h.unsqueeze(1))
        
        # outputs: [batch_size, seq_length, hidden_dim]
        outputs = torch.cat(outputs, dim=1)
        
        return outputs, h

### Классификатор с собственным GRU

Создаём нейросеть для классификации с использованием собственной реализации GRU.

In [ ]:
class CustomGRUClassifier(nn.Module):
    """Классификатор с собственной реализацией GRU"""
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(CustomGRUClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.custom_gru = CustomGRULayer(embedding_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # x: [batch_size, seq_length]
        embedded = self.embedding(x)  # [batch_size, seq_length, embedding_dim]
        output, hidden = self.custom_gru(embedded)  # hidden: [batch_size, hidden_dim]
        logits = self.fc(hidden)  # [batch_size, output_dim]
        return logits

# Создание и обучение модели с собственным GRU
custom_gru_model = CustomGRUClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM)
print(custom_gru_model)

optimizer_custom_gru = optim.Adam(custom_gru_model.parameters(), lr=0.001)

print("\nОбучение модели с собственным GRU...")
custom_gru_losses = train_model(custom_gru_model, train_loader, criterion, optimizer_custom_gru, NUM_EPOCHS)

### Оценка качества модели с собственным GRU

In [ ]:
# Оценка модели с собственным GRU
custom_gru_acc, custom_gru_prec, custom_gru_rec, custom_gru_f1, custom_gru_labels, custom_gru_preds = evaluate_model(custom_gru_model, test_loader)

print(f"Custom GRU - Accuracy: {custom_gru_acc:.4f}")
print(f"Custom GRU - Precision: {custom_gru_prec:.4f}")
print(f"Custom GRU - Recall: {custom_gru_rec:.4f}")
print(f"Custom GRU - F1: {custom_gru_f1:.4f}")
print("\nClassification Report:")
print(classification_report(custom_gru_labels, custom_gru_preds, target_names=label_encoder.classes_))

## 11. Графики обучения и итоговый вывод

Строим графики loss по эпохам для всех моделей и делаем итоговый вывод о работе моделей.

In [ ]:
# Графики loss по эпохам
plt.figure(figsize=(12, 6))
plt.plot(rnn_losses, label='RNN', marker='o')
plt.plot(lstm_losses, label='LSTM', marker='s')
plt.plot(gru_losses, label='GRU', marker='^')
plt.plot(custom_gru_losses, label='Custom GRU', marker='d')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Графики обучения (Loss по эпохам)')
plt.legend()
plt.grid(True)
plt.show()

# Итоговая сводная таблица с собственным GRU
final_comparison = pd.DataFrame({
    'Model': ['RNN', 'LSTM', 'GRU', 'Custom GRU'],
    'Accuracy': [rnn_acc, lstm_acc, gru_acc, custom_gru_acc],
    'Precision': [rnn_prec, lstm_prec, gru_prec, custom_gru_prec],
    'Recall': [rnn_rec, lstm_rec, gru_rec, custom_gru_rec],
    'F1': [rnn_f1, lstm_f1, gru_f1, custom_gru_f1]
})

print("\nИтоговое сравнение всех моделей:")
print(final_comparison.to_string(index=False))

# График итогового сравнения
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, metric in enumerate(metrics):
    axes[i].bar(final_comparison['Model'], final_comparison[metric], color=['blue', 'green', 'red', 'orange'])
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    axes[i].set_ylabel('Score')
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print("\n=== ИТОГОВЫЙ ВЫВОД ===")
print(f"Лучшая модель по F1: {final_comparison.loc[final_comparison['F1'].idxmax(), 'Model']}")
print(f"Собственная реализация GRU показала F1: {custom_gru_f1:.4f}")
print("Собственный GRU-блок реализован корректно и может использоваться для классификации текстов.")